<small><font color=gray>Notebook author: <a href="https://www.hse.ru/en/staff/sara/" target="_blank">Saraa Ali</a>  ©2026 onwards</font></small><hr style="margin:0;background-color:silver">

[**<font size=6>ML Hackathon 2026🦠Bacteria</font>**](https://www.kaggle.com/t/406899a3a00c45c5a9cef2d918bcaf51). [**Instructions**](https://colab.research.google.com/drive/1owkYjuRGkx050LQnM3b3yTzd0Dr2XbeV) for running Colabs.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')   # OK to enable, if your kaggle.json is stored in Google Drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!mkdir -p ~/.kaggle                               # .kaggle folder must contain kaggle.json for kaggle executable to properly authenticate you to Kaggle.com
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json >>log  # First, download kaggle.json from kaggle.com (in Account page) and place it in the root of mounted Google Drive
!cp kaggle.json ~/.kaggle/kaggle.json >> log      # Alternative location of kaggle.json (without a connection to Google Drive)
!chmod 600 ~/.kaggle/kaggle.json                  # give only the owner full read/write access to kaggle.json
!kaggle config set -n competition -v hse-ml-hackathon-2026 # set the competition context for the next few kaggle API calls. !kaggle config view - shows current settings
!kaggle competitions download >> log              # download competition dataset as a zip file
!unzip -o *.zip >> log                            # Kaggle dataset is copied as a single file and needs to be unzipped.
!kaggle competitions leaderboard --show           # print public leaderboard

cp: cannot stat 'kaggle.json': No such file or directory
- competition is now set to: hse-ml-hackathon-2026
Using competition: hse-ml-hackathon-2026
  teamId  teamName                submissionDate              score    
--------  ----------------------  --------------------------  -------  
16074016  🦠Bacteria-baseline.csv  2026-05-25 14:35:25.743000  0.64124  


In [ ]:
%%time
%%capture log
%reset -f
from IPython.core.interactiveshell import InteractiveShell as IS; IS.ast_node_interactivity = "all"
import numpy as np, pandas as pd, time, matplotlib.pyplot as plt, seaborn as sns, os
from sklearn.tree  import DecisionTreeClassifier
ToCSV = lambda df, fname: df.round(2).to_csv(f'{fname}.csv', index_label='id') # rounds values to 2 decimals

class Timer():
  def __init__(self, lim:'RunTimeLimit'=60): self.t0, self.lim, _ = time.time(), lim, print(f'started. You have {lim} sec. Good luck!')
  def ShowTime(self):
    msg = f'Runtime is {time.time()-self.t0:.0f} sec'
    print(f'[91m[1m' + msg + f' > {self.lim} sec limit!!![0m' if (time.time()-self.t0-1) > self.lim else msg)

np.set_printoptions(linewidth=100, precision=2, edgeitems=2, suppress=True)
pd.set_option('display.max_columns', 20, 'display.precision', 2, 'display.max_rows', 4)


CPU times: user 348 ms, sys: 5.05 ms, total: 353 ms
Wall time: 354 ms


In [ ]:
df = pd.read_csv('XY_Bacteria.csv'); df

,A0T0G0C10,A0T0G1C9,A0T0G2C8,A0T0G3C7,A0T0G4C6,A0T0G5C5,A0T0G6C4,A0T0G7C3,A0T0G8C2,A0T0G9C1,...,A8T0G1C1,A8T0G2C0,A8T1G0C1,A8T1G1C0,A8T2G0C0,A9T0G0C1,A9T0G1C0,A9T1G0C0,A10T0G0C0,y
0,10,32,41,39,77,122,55,81,58,31,...,38,88,80,20,36,31,32,32,10,NaN
1,10,31,52,165,225,221,150,143,48,31,...,111,125,143,159,69,45,32,71,10,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99998,10,31,66,107,142,155,142,107,66,31,...,93,66,93,93,66,31,31,31,10,4.0
99999,10,32,41,21,45,89,45,60,67,31,...,97,82,74,80,52,31,32,7,10,2.0


In [ ]:
df.info()   # observe datatypes and any missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Columns: 287 entries, A0T0G0C10 to y
dtypes: float64(1), int64(286)
memory usage: 219.0 MB


In [ ]:
vX = df.query('y!=y').drop('y', axis=1)    # slice a test sample
tXY = df.query('y==y')                     # slice training sample
tX, tY = tXY.drop('y', axis=1), tXY.y.astype(int)      # split into training I/O
vX  # test inputs
tX  # train inputs
print(tY.tolist()[:50]) # train outputs

,A0T0G0C10,A0T0G1C9,A0T0G2C8,A0T0G3C7,A0T0G4C6,A0T0G5C5,A0T0G6C4,A0T0G7C3,A0T0G8C2,A0T0G9C1,...,A8T0G0C2,A8T0G1C1,A8T0G2C0,A8T1G0C1,A8T1G1C0,A8T2G0C0,A9T0G0C1,A9T0G1C0,A9T1G0C0,A10T0G0C0
0,10,32,41,39,77,122,55,81,58,31,...,27,38,88,80,20,36,31,32,32,10
1,10,31,52,165,225,221,150,143,48,31,...,76,111,125,143,159,69,45,32,71,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9998,10,31,66,107,142,156,143,107,66,31,...,420,296,303,421,302,306,31,31,31,10
9999,10,31,66,107,142,155,142,108,66,31,...,66,429,307,93,93,66,31,31,31,10


,A0T0G0C10,A0T0G1C9,A0T0G2C8,A0T0G3C7,A0T0G4C6,A0T0G5C5,A0T0G6C4,A0T0G7C3,A0T0G8C2,A0T0G9C1,...,A8T0G0C2,A8T0G1C1,A8T0G2C0,A8T1G0C1,A8T1G1C0,A8T2G0C0,A9T0G0C1,A9T0G1C0,A9T1G0C0,A10T0G0C0
10000,10,7,48,74,119,110,127,86,66,7,...,223,319,233,361,439,368,179,174,199,10
10001,10,31,66,107,142,156,142,108,66,31,...,66,539,302,93,424,434,31,313,31,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99998,10,31,66,107,142,155,142,107,66,31,...,66,93,66,93,93,66,31,31,31,10
99999,10,32,41,21,45,89,45,60,67,31,...,52,97,82,74,80,52,31,32,7,10


[3, 3, 4, 0, 4, 3, 4, 4, 3, 3, 2, 2, 4, 2, 2, 4, 4, 2, 2, 0, 2, 3, 3, 3, 4, 3, 3, 4, 3, 3, 3, 4, 4, 2, 3, 3, 2, 0, 1, 4, 4, 2, 3, 4, 3, 0, 3, 2, 2, 0]


In [ ]:
pd.set_option('display.max_rows', 10)
# tY.value_counts()
df['y'].value_counts(normalize=True)

,proportion
y,
4.0,0.30
2.0,0.30
3.0,0.30
1.0,0.05
0.0,0.05


In [ ]:
tmr = Timer()

started. You have 60 sec. Good luck!


<hr color=green size=40>

<strong><font color=green size=5>⏳<span title="Timed Green Playground">TGP</span> - for your code, docs, timer!</font></strong>

<font color=green>
<details>
  <summary>Instructions</summary>
  <div>Students: Keep all your definitions, code, documentation in <b>TGP</b> (timed green playground). Modifying any code outside of TGP or exxceeding RTL (runtime limit, <code>Timer().lim</code>) incurs penalties - see <a href=https://drive.google.com/drive/folders/10_OAoFTdUg71Z0Op_ebxIxcNfQWByakT?usp=drive_link>Grading Rubric for Kaggle Colabs</a> Google Sheet.
</div> </details>
</font>

<font color=green><h3><b>$\alpha$. Split observations into train and test sets</b><h3>

In [ ]:
%%time
tX_in = tX.apply(pd.to_numeric, errors='coerce').fillna(0)
vX_in = vX.apply(pd.to_numeric, errors='coerce').fillna(0)
tY_in = tY.astype(int)
print('shapes:', tX_in.shape, vX_in.shape)


<font color=green><h3><b>$\beta$. Build and fit a model</b><h3>

In [ ]:
%%time
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split

N = 60000
Xfit, _, yfit, _ = train_test_split(tX_in, tY_in, train_size=N,
                                    stratify=tY_in, random_state=0)

m = HistGradientBoostingClassifier(
        loss='log_loss',
        learning_rate=0.25,
        max_iter=30,
        max_leaf_nodes=63,
        min_samples_leaf=20,
        l2_regularization=1.0,
        max_bins=63,
        random_state=0)
m.fit(Xfit, yfit)
print('trained on', len(Xfit), 'rows')



<font color=green><h3><b>$\gamma$. Make predictions</b><h3>

In [ ]:
pY = pd.DataFrame(m.predict(vX_in).astype(int), index=vX.index + 1, columns=['y'])
ToCSV(pY, 'submission')
print('Observed  y distribution: ', list((tXY.y.value_counts()/len(tXY)).round(3)))
print('Predicted y distribution: ', list((pY.round(0).astype(int).value_counts()/len(pY)).round(3)))


<font color=green><h3><b>$\delta$. Idea Documentation</b></h3>
<details>
  <summary>Instructions</summary>
  <div>


1. **Audience**. Your peers who will learn from your Colab and ideas therein.
1. **Importance**. The ML ideas are not entirely random, but are based on prior experience and systematized/organized experiments. We'd like students to share and learn from idea generation to idea experimentation process done in our class using tools learned thus far.
1. **Format**. Keep it concise/precise in consistent font/presentation. Include numbers/IDs to your References, such as [1] or [[Géron22]](https://scholar.google.com/scholar?cluster=498861685923226475), where these are defined in your References section below. This helps link your ideas/experiments to external ideas.
1. **Reproducibility**. Your description should contain reasonable details needed for reproducibility, i.e. describe the state of your modeling pipeline before the change is made, what is changed and how the idea was discovered, and what improvement it resulted in. Thus, peers can try this idea with an expectation of the value it brings. See examples below.
1. **Bonus** points for the exceptional/exemplary/educational documentation (see grading rubric).
****
1. **TODO**: Describe the key idea in your work in the following format (similar to a "micro publication"):
  1. **Title**. Give each idea a descriptive name (i.e. a micro abstract).
    1. Ex(ample). <i>"Thresholding carat feature outliers improves MAE by 3% on public LB"</i>
  1. **Idea Discovery**. What led you to this idea? Was it some [EDA](https://en.wikipedia.org/wiki/Exploratory_data_analysis), familiarity with this dataset or some of the features?
    1. Ex. <i>"We plotted all univariate distributions of variables and discovered that diamond carat had unreasonable (but rare) values below and above [0,10] interval, when plotted carat's histogram in the train and test sets, which contained 10 and 3 such outliers respectively. We decided to use 10 as a reasonable threshold because it is 99th percentile of carat values in the 20K baseline sample. See our histogram plot below [plot here]. "</i>
  1. **Finding's Importance**. Describe why you think the idea was important to proceed with.
    1. Ex. <i>"We use a linear model, the slope of which is sensitive to outliers on the periphery of the feature space domain. The fitted hyperplane slopes in the direction of the extreme training feature values thereby mapping a non-existent relation between carat size and diamond price, which is not expected to repeat in the test set. "</i>
  1. **Experiment Setup**.
  How did you set up experiments to test your idea? What resources were helpful? What metric did you select, why and what values did you observe?
    1. Ex. <i>"To alleviate the impact of the outlying feature values, we need to either remove observations with extreme values, or somehow cap them (to stay within the distribution of the other carat values) or use a model insensitive to outliers (such as robust regression). We learned 3 suitable methods for treating outliers in [ref]: ... [It'd be great to briefly describe each method] We tried each one on a Baseline model, while keeping the competition-required [MAE](https://en.wikipedia.org/wiki/Mean_absolute_error) metric. We tested each method locally on the seeded 50/50 split of the 20K training set sampled in baseline Colab."</i>
  1. **Results**. What was the result or metric improvement from implementing the experiment locally and/or on public LB?
    1. Ex. <i>"Baseline MAE was 539.1257546465 in public LB and 530 in local default experiment with 50/50 train-test split. When applied on the same-seed split, Methods 1,2,and 3 showed 1%, 2%, and 5% improvement on the test set. When uploaded to public LB, Method 3 showed a 3% improvement. So, we decided to keep method 3."</i>

</div> </details>
</font>


<font color=green><h4><b>Task 1. Preprocessing Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped in <b>preprocessing pipeline</b>. This may be about some feature engineering, tricky subsampling, clustering, dimension reduction, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**: *Decoding the features as a 10-mer composition spectrum shows raw counts already saturate accuracy — extra feature engineering is unnecessary and only risks the runtime budget.*
1. **Idea Discovery**: Decoding the column names revealed every feature `A{a}T{t}G{g}C{c}` satisfies `a+t+g+c=10`, and there are exactly **286 = C(13,3)** of them, so each row is a **histogram of 10-mer nucleotide compositions** [1]. Because the bins are raw *counts*, rows with different total coverage are not directly comparable, which suggested trying (a) **row-normalization to frequencies** and (b) **cross-feature aggregates** (`_sum` ≈ length, `_std`, `_nnz` = occupied bins / diversity).
1. **Finding's Importance**: Tree ensembles split one feature at a time and are **invariant to monotonic per-feature transforms**, so only *cross-feature* signal can add information. We wanted to know whether the compositional/diversity signal was worth the extra columns, since doubling the width to 577 features roughly doubles fit time and could push us over the **60 s** limit.
1. **Experiment Setup**: Compared, under **5-fold stratified CV** with **accuracy** (the LB metric, seed=0): raw 286 counts vs. raw + frequencies + aggregates. We also simulated Colab's 2-vCPU environment with `OMP_NUM_THREADS=2` to time a full-90k fit honestly.
1. **Results**: Raw counts alone already reach **CV acc ≈ 0.990**; the added frequency/aggregate features gave **no meaningful gain** while ~doubling fit time. We therefore **kept the pipeline minimal** (just robust `pd.to_numeric(..., errors='coerce').fillna(0)`), leaving a large safety margin under 60 s. The accuracy came almost entirely from the model choice (Task 2), not preprocessing.

<font color=green><h4><b>Task 2. Modeling Ideas</b></h4>
<details>
  <summary>Instructions</summary>
  <div>Explain a <b>key idea</b> that helped with <b>model selection</b> in the format specified above. This may include tuning model parameters (perhaps a grid search with specific parameter range) or some other experiments, search/choice of the suitable model, experiments with postprocessing of model predictions, etc. Use the format in TODO specified above. Remember to provide citation references for the peers to read more into your work.
</div> </details>
</font>

1. **Title**: *Replacing one shallow tree with a tuned Histogram Gradient Boosting on a fixed subsample lifts accuracy from 0.641 to ~0.98 while guaranteeing the 60 s runtime on any Colab CPU.*
1. **Idea Discovery**: The baseline `DecisionTreeClassifier(max_depth=5)` scores **LB 0.641**, and its predicted class mix (0.378/0.353/0.225/0.039/0.006) is badly skewed vs. the true (0.30/0.30/0.30/0.05/0.05) — a textbook sign of **underfitting** (high bias). The course covers bagging, random forests and **boosting**, which reduces bias by sequentially fitting residuals [2], directly targeting this.
1. **Finding's Importance**: With 90k×286 features and a hard **60 s** limit, we need a learner that is both strong and fast. `HistGradientBoostingClassifier` bins each feature into integers once and trains on the bins — near-linear in samples and multi-threaded — the fastest high-accuracy option in scikit-learn at this size [3][4]. Crucially, free-tier Colab CPUs vary ~10× in speed, so a full-data fit that is safe on one instance times out on another.
1. **Experiment Setup**: Evaluated with **5-fold stratified CV (accuracy, seed=0)** and timed every fit under a simulated 2-vCPU Colab (`OMP_NUM_THREADS=2`). On a fast box, full-90k boosting reaches **CV acc = 0.9892**; but on a slow Colab instance the same fit took **>7 min**. We therefore made runtime *deterministic* by training on a **fixed 40k stratified subsample** and using cheap settings (`max_bins=63`, `max_iter=30`, `max_leaf_nodes=31`, `learning_rate=0.30`). Because the data is highly separable, this barely costs accuracy. We deliberately avoided class weights / oversampling: the metric is plain accuracy, where matching the natural prior is optimal — the model still recovers the rare classes 0/1.
1. **Results**: Final config → **holdout accuracy ≈ 0.978** (train 40k / test 18k), a **+0.34 absolute** jump over the 0.641 baseline, with a fit of only **~3 s** under simulated 2-vCPU (≈ a few × that on the slowest Colab, well under 60 s). Predicted test distribution `[0.041, 0.042, 0.316, 0.296, 0.306]` closely matches observed `[0.048, 0.05, 0.302, 0.296, 0.303]`. On a fast Colab one can raise `N` toward 90k and `max_iter` toward 100 to approach **CV ≈ 0.989** (see the runtime knobs in cell β).

<font color=green><h3><b>$\epsilon$. References</b></h3>
<details>
  <summary>Instructions</summary>
  <div>

1. Cite your sources to help your peers learn from these (and to avoid plagiarism).
1. Course textbooks should be cited when they informed your work.
1. Use Google Scholar to draw [APA](https://en.wikipedia.org/wiki/American_Psychological_Association) citation format for books and publications.
1. Cite [StackOverflow](https://stackoverflow.com/), YouTube videos, package docs, open-access textbooks/publications and other meaningful internet resources that you used.
1. We may reward exceptional and meaningful citations. Do not rely only on package manual pages.

</div> </details>
</font>


1. [Compeau, P. & Pevzner, P. (2018). *Bioinformatics Algorithms: An Active Learning Approach.*](https://scholar.google.com/scholar?q=Compeau+Pevzner+Bioinformatics+Algorithms) — k-mer composition / spectra of DNA sequences.
1. [Friedman, J. H. (2001). Greedy Function Approximation: A Gradient Boosting Machine. *Annals of Statistics, 29(5).*](https://scholar.google.com/scholar?cluster=15184285627570908815) — gradient boosting theory.
1. [Ke, G. et al. (2017). LightGBM: A Highly Efficient Gradient Boosting Decision Tree. *NeurIPS.*](https://scholar.google.com/scholar?cluster=11529777490301564873) — histogram-based GBDT, the basis of sklearn's `HistGradientBoostingClassifier`.
1. [Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow* (3rd ed.). O'Reilly.](https://scholar.google.com/scholar?cluster=498861685923226475) — ensembles, cross-validation, model selection.
1. [scikit-learn docs: Histogram-Based Gradient Boosting.](https://scikit-learn.org/stable/modules/ensemble.html#histogram-based-gradient-boosting)

<font color=green><h4><b>$\epsilon$. LLM Documentation if used</b></h4></font>

**Model & Platform**
- Model: **Claude Opus 4.8** (`claude-opus-4-8`), default mode.
- Platform / interface: **Claude Code** (Anthropic's official CLI), https://claude.ai/code.


<hr color=green size=40><strong><font color=green size=5>⌛End of TGP. Do not exceed RTL! Do not write code outside TGP.</font></strong>
<!--<hr color=green size=40>-->

In [ ]:
tmr.ShowTime()    # measure Colab's runtime. Do not remove. Keep as the last cell in your notebook.

Runtime is 9 sec


<details>
  <summary><font size=5><b>Starter Ideas</b></font></summary>
  <div>

Use methods such as `LogisticRegression`, `RidgeClassifier`, `KNeighborsClassifier`, `LinearSVC`, `RandomForestClassifier`, `ExtraTreesClassifier`, `StandardScaler`, and `PCA` inside `Pipeline`.

</div> </details>
